# バイオ技術 2-4：AI研究チャレンジ

## 声の特徴からParkinson病を判別できるか？
### 高いAccuracyは本当に信頼できる？

午前中は、

- Classification
- Regression
- Decision Tree
- CNN
- Train/Test split
- Accuracy
- F1-score
- ROC-AUC
- Confusion Matrix

などを学びました。

午後は、実際の公開研究データを使い、

> **AIの性能評価は、データの分け方によって変わるのか？**

を調べます。

---

## 今日の研究上の問い

Parkinson病では、発声にも変化が現れることがあります。

では、

> **声から計算した特徴量だけで、Parkinson病を判別できるでしょうか？**

さらに、

> **同じ人から複数の録音があるデータでは、普通のrandom splitを使ってよいのでしょうか？**

を考えます。

---

## 今日のゴール

1. 公開研究データを読み込む
2. 音声特徴量を可視化する
3. Logistic RegressionでParkinson病を分類する
4. recording-level random splitで性能を評価する
5. Train/Test間のsubject overlapを確認する
6. subject-level splitで再評価する
7. 2つの評価方法を比較する
8. group-aware cross-validationを体験する
9. 高いAccuracyをそのまま信じてよいか考える

---

## 今日の流れ（約2時間）

| 時間 | 内容 |
|---|---|
| 10分 | 論文と研究背景 |
| 15分 | データ構造の確認 |
| 15分 | 音声特徴量の可視化 |
| 20分 | recording-level splitで分類 |
| 15分 | subject overlapを確認 |
| 20分 | subject-level splitで再解析 |
| 15分 | Cross-validationで比較 |
| 10分 | 論文と比較・考察 |


---
# 0. 原著論文

今回の題材は、次の研究です。

**Little MA, McSharry PE, Hunter EJ, Spielman J, Ramig LO.  
Suitability of dysphonia measurements for telemonitoring of Parkinson's disease.  
IEEE Transactions on Biomedical Engineering. 2009;56(4):1015–1022.**

論文では、Parkinson病患者と健常者の持続発声から音声特徴量を計算し、
Parkinson病の判別可能性を検討しています。

論文では、

- 相関の低い特徴量を選択
- 特徴量の組合せを探索
- kernel SVMを使用

し、4つの特徴量の組合せで高い分類性能を報告しています。

### 今日は完全再現ではありません

このNotebookでは、

> **同じ公開データを使って、AI評価設計の重要性を考える**

ことを目的とします。

原著論文の完全な手法再現ではありません。


---
# 1. ライブラリを読み込む


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve
)

print("Libraries loaded.")


---
# 2. UCI Parkinsons Datasetを読み込む

UCI Machine Learning Repositoryで公開されているデータを使います。

各行は**1回の音声録音**です。

目的変数`status`は、

- 0：Healthy
- 1：Parkinson's disease

です。


In [ ]:
DATA_URL = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/parkinsons/"
    "parkinsons.data"
)

df = pd.read_csv(DATA_URL)

print("Data shape:", df.shape)
display(df.head())


### ミニ確認1

次の問いに答えてください。

1. 行数はいくつですか？
2. 列数はいくつですか？
3. 1行は「1人」ですか、それとも「1録音」ですか？
4. 目的変数の列名は何ですか？

答え：

1.
2.
3.
4.


---
# 3. 目的変数を確認する

まず、recording単位でHealthy / Parkinson's diseaseの数を確認します。


In [ ]:
df["diagnosis"] = df["status"].map({
    0: "Healthy",
    1: "Parkinson"
})

recording_counts = (
    df["diagnosis"]
    .value_counts()
    .rename_axis("diagnosis")
    .reset_index(name="recordings")
)

display(recording_counts)


In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="diagnosis",
    order=["Healthy", "Parkinson"]
)

plt.xlabel("Diagnosis")
plt.ylabel("Number of recordings")
plt.title("Class Distribution at Recording Level")
plt.show()


### 注意

このグラフは**人の数**ではありません。

同じ人から複数の録音があるため、

> recording数 ≠ subject数

です。

ここが今日の重要なポイントです。


---
# 4. `name`列からsubject IDを取り出す

例えば、

```text
phon_R01_S01_1
phon_R01_S01_2
phon_R01_S01_3
```

は、同じsubjectからの別録音です。

最後の`_1`, `_2`, `_3` ... を除き、
grouping用の`subject_id`を作ります。


In [ ]:
df["subject_id"] = df["name"].str.replace(
    r"_\d+$",
    "",
    regex=True
)

df["recording_no"] = df["name"].str.extract(
    r"_(\d+)$"
).astype(int)

display(
    df[
        [
            "name",
            "subject_id",
            "recording_no",
            "diagnosis"
        ]
    ].head(12)
)


In [ ]:
print(
    "Number of recordings:",
    len(df)
)

print(
    "Unique grouping IDs:",
    df["subject_id"].nunique()
)

recordings_per_subject = (
    df.groupby("subject_id")
    .size()
    .sort_values()
)

display(
    recordings_per_subject.to_frame(
        "recordings"
    )
)


### データの注意

UCIのデータ説明では31 peopleと記載されていますが、
現在のCSVについて、`name`の末尾の録音番号を除いて数えると、
32個のgrouping IDが得られます。

この演習では、

> **同じ`subject_id`を持つ録音をTrainとTestに分けない**

ことを目的とし、CSV中のIDをそのままgroupとして使います。

公開データを使うときは、

> 論文の記載、repositoryのmetadata、実際のファイル構造

を自分で確認することも重要です。


---
# 5. 音声特徴量を見る

このデータには、例えば次の特徴量があります。

- `MDVP:Jitter(%)`：基本周波数の変動
- `MDVP:Shimmer`：振幅の変動
- `HNR`：harmonics-to-noise ratio
- `RPDE`：nonlinear dynamical complexity measure
- `DFA`：fractal scaling exponent
- `PPE`：pitch period entropy

まず、いくつかの特徴量を可視化します。


In [ ]:
SELECTED_FEATURES = [
    "MDVP:Jitter(%)",
    "MDVP:Shimmer",
    "HNR",
    "RPDE",
    "DFA",
    "PPE"
]

long_df = df.melt(
    id_vars=[
        "diagnosis",
        "subject_id"
    ],
    value_vars=SELECTED_FEATURES,
    var_name="feature",
    value_name="value"
)

g = sns.catplot(
    data=long_df,
    x="diagnosis",
    y="value",
    col="feature",
    col_wrap=3,
    kind="box",
    sharey=False,
    height=3.2
)

g.set_titles("{col_name}")
g.set_axis_labels("", "Value")

plt.tight_layout()
plt.show()


### ミニ演習2：特徴量を観察する

1. HealthyとParkinsonで違いがありそうな特徴量はどれですか？
2. 分布の重なりが大きい特徴量はありますか？
3. 1つの特徴量だけで完全に分類できそうですか？

答え：

1.
2.
3.


---
# 6. 各特徴量とstatusの関係を見る

各特徴量と`status`の相関係数を計算します。

これは候補特徴量を見るための簡単な探索です。

### 注意

相関係数が大きいことは、

> その特徴量がParkinson病の原因である

ことを意味しません。


In [ ]:
feature_columns = [
    col
    for col in df.columns
    if col not in [
        "name",
        "status",
        "diagnosis",
        "subject_id",
        "recording_no"
    ]
]

correlations = (
    df[feature_columns + ["status"]]
    .corr(numeric_only=True)["status"]
    .drop("status")
    .sort_values(
        key=np.abs,
        ascending=False
    )
)

display(
    correlations
    .head(15)
    .to_frame("correlation_with_status")
)


In [ ]:
top_corr = (
    correlations
    .head(12)
    .sort_values()
)

plt.figure(figsize=(8, 6))

top_corr.plot(
    kind="barh"
)

plt.xlabel("Correlation with status")
plt.title("Top Feature Correlations with Status")
plt.tight_layout()
plt.show()


### 考えてみよう

相関係数の絶対値が大きい特徴量と、
先ほどboxplotで差がありそうに見えた特徴量は一致していますか？

答え：


---
# 7. 機械学習用データを準備する

- `X`：音声特徴量
- `y`：status
- `groups`：subject_id

を作ります。


In [ ]:
X = df[feature_columns].copy()
y = df["status"].copy()
groups = df["subject_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of groups:", groups.nunique())


---
# Part 1：Recording-level random split

まず、午前と同じように、

```python
train_test_split()
```

を使います。

この時点では、subject_idを分割条件には使いません。

つまり、

> **録音を1行ずつ独立なdataとしてrandom splitする**

方法です。


---
# 8. Recording-level split

70%をTraining data、30%をTest dataにします。


In [ ]:
X_train_rec, X_test_rec, y_train_rec, y_test_rec = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training recordings:", len(X_train_rec))
print("Test recordings    :", len(X_test_rec))

print()
print("Test class distribution:")
print(y_test_rec.value_counts())


---
# 9. Logistic Regressionを学習する

今回は、分割方法の違いを比較することが目的なので、
モデルは同じLogistic Regressionを使います。

StandardScalerとLogistic RegressionをPipelineでつなぎます。


In [ ]:
recording_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])

recording_model.fit(
    X_train_rec,
    y_train_rec
)

recording_pred = recording_model.predict(
    X_test_rec
)

recording_prob = recording_model.predict_proba(
    X_test_rec
)[:, 1]


---
# 10. Recording-level splitの性能を評価する


In [ ]:
recording_accuracy = accuracy_score(
    y_test_rec,
    recording_pred
)

recording_f1 = f1_score(
    y_test_rec,
    recording_pred
)

recording_auc = roc_auc_score(
    y_test_rec,
    recording_prob
)

print(f"Accuracy: {recording_accuracy:.3f}")
print(f"F1-score: {recording_f1:.3f}")
print(f"ROC-AUC : {recording_auc:.3f}")


In [ ]:
cm_rec = confusion_matrix(
    y_test_rec,
    recording_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rec,
    display_labels=[
        "Healthy",
        "Parkinson"
    ]
)

disp.plot(
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title(
    "Recording-level Split"
)

plt.show()


In [ ]:
fpr_rec, tpr_rec, _ = roc_curve(
    y_test_rec,
    recording_prob
)

plt.figure(figsize=(6, 5))

plt.plot(
    fpr_rec,
    tpr_rec,
    label=f"AUC = {recording_auc:.3f}"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Recording-level Split")
plt.legend()
plt.show()


### ミニ演習3：最初の結果を評価する

1. Accuracyはいくつでしたか？
2. F1-scoreはいくつでしたか？
3. ROC-AUCはいくつでしたか？
4. この結果を見て「よいAIができた」と言ってよいでしょうか？

答え：

1.
2.
3.
4.


---
# 11. TrainとTestに同じsubjectが入っていないか？

ここで、TrainとTestのsubject_idを調べます。

recording-level random splitでは、

> 同じ人の録音がTrainとTestの両方に入る可能性

があります。


In [ ]:
train_subjects_rec = set(
    groups.loc[X_train_rec.index]
)

test_subjects_rec = set(
    groups.loc[X_test_rec.index]
)

overlap_subjects_rec = (
    train_subjects_rec
    & test_subjects_rec
)

print(
    "Train subjects:",
    len(train_subjects_rec)
)

print(
    "Test subjects:",
    len(test_subjects_rec)
)

print(
    "Overlapping subjects:",
    len(overlap_subjects_rec)
)

print()
print("Overlap:")
print(
    sorted(overlap_subjects_rec)
)


## 重要な問い

もし同じsubjectの、

```text
Recording 1 → Training data
Recording 2 → Test data
```

となっていたら、

> **未知の人に対する性能を評価している**

と言えるでしょうか？

モデルが病気の特徴だけでなく、
同じ人に固有の声の特徴を利用している可能性も考える必要があります。

これを確かめるため、次はsubject単位で分けます。


---
# Part 2：Subject-level split

次は、

> **あるsubjectの全録音をTrainまたはTestのどちらか一方だけに入れる**

ように分割します。

これにより、

```text
Subject A → Train only
Subject B → Train only
Subject C → Test only
```

となります。


---
# 12. GroupShuffleSplitでsubject単位に分ける


In [ ]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx_group, test_idx_group = next(
    group_split.split(
        X,
        y,
        groups=groups
    )
)

X_train_group = X.iloc[
    train_idx_group
]

X_test_group = X.iloc[
    test_idx_group
]

y_train_group = y.iloc[
    train_idx_group
]

y_test_group = y.iloc[
    test_idx_group
]

groups_train = groups.iloc[
    train_idx_group
]

groups_test = groups.iloc[
    test_idx_group
]

print("Training recordings:", len(X_train_group))
print("Test recordings    :", len(X_test_group))

print()
print("Training subjects:", groups_train.nunique())
print("Test subjects    :", groups_test.nunique())

print()
print("Test class distribution:")
print(y_test_group.value_counts())


---
# 13. subject overlapが0であることを確認する


In [ ]:
overlap_subjects_group = (
    set(groups_train)
    & set(groups_test)
)

print(
    "Overlapping subjects:",
    len(overlap_subjects_group)
)

print(
    "Overlap:",
    overlap_subjects_group
)


`Overlapping subjects: 0`になっていることを確認してください。


---
# 14. 同じLogistic Regressionで再評価する

モデルは変えません。

変えるのは**データの分け方だけ**です。


In [ ]:
group_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])

group_model.fit(
    X_train_group,
    y_train_group
)

group_pred = group_model.predict(
    X_test_group
)

group_prob = group_model.predict_proba(
    X_test_group
)[:, 1]


In [ ]:
group_accuracy = accuracy_score(
    y_test_group,
    group_pred
)

group_f1 = f1_score(
    y_test_group,
    group_pred
)

group_auc = roc_auc_score(
    y_test_group,
    group_prob
)

print(f"Accuracy: {group_accuracy:.3f}")
print(f"F1-score: {group_f1:.3f}")
print(f"ROC-AUC : {group_auc:.3f}")


In [ ]:
cm_group = confusion_matrix(
    y_test_group,
    group_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_group,
    display_labels=[
        "Healthy",
        "Parkinson"
    ]
)

disp.plot(
    cmap="Blues",
    values_format="d",
    colorbar=False
)

plt.title(
    "Subject-level Split"
)

plt.show()


---
# 15. 2つの分割方法を比較する


In [ ]:
comparison = pd.DataFrame({
    "Split method": [
        "Recording-level",
        "Subject-level"
    ],
    "Accuracy": [
        recording_accuracy,
        group_accuracy
    ],
    "F1": [
        recording_f1,
        group_f1
    ],
    "ROC-AUC": [
        recording_auc,
        group_auc
    ],
    "Subject overlap": [
        len(overlap_subjects_rec),
        len(overlap_subjects_group)
    ]
})

display(comparison)


In [ ]:
comparison_long = comparison.melt(
    id_vars="Split method",
    value_vars=[
        "Accuracy",
        "F1",
        "ROC-AUC"
    ],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=comparison_long,
    x="Metric",
    y="Score",
    hue="Split method"
)

plt.ylim(0, 1)
plt.title("Performance by Split Method")
plt.show()


### ミニ演習4：結果を比較する

| Metric | Recording-level | Subject-level |
|---|---:|---:|
| Accuracy | | |
| F1 | | |
| ROC-AUC | | |

### 考えてみよう

1. 分割方法だけで性能は変わりましたか？
2. どちらが「未知の人への性能」に近い評価だと思いますか？
3. recording-level splitで性能が高く見える理由として何が考えられますか？
4. Test setの構成によって結果が変わる可能性はありますか？

答え：

1.
2.
3.
4.


---
# 16. 1回のsplitだけで結論してよいか？

subject数が少ない場合、

> たまたま誰がTest dataに入ったか

によって性能が変わる可能性があります。

そこで、5-fold cross-validationを行います。

比較するのは、

### Recording-level CV

```text
StratifiedKFold
```

録音単位でfoldを作ります。

### Subject-level CV

```text
StratifiedGroupKFold
```

同じsubjectの録音を同じfoldに保ちます。


In [ ]:
cv_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000
        )
    )
])

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc"
}


### Recording-level 5-fold CV


In [ ]:
recording_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

recording_cv_results = cross_validate(
    cv_model,
    X,
    y,
    cv=recording_cv,
    scoring=scoring
)

recording_cv_results


### Subject-level 5-fold CV


In [ ]:
subject_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

subject_cv_results = cross_validate(
    cv_model,
    X,
    y,
    groups=groups,
    cv=subject_cv,
    scoring=scoring
)

subject_cv_results


---
# 17. Cross-validation結果をまとめる


In [ ]:
cv_summary = pd.DataFrame({
    "Split method": [
        "Recording-level CV",
        "Subject-level CV"
    ],
    "Accuracy mean": [
        recording_cv_results[
            "test_accuracy"
        ].mean(),
        subject_cv_results[
            "test_accuracy"
        ].mean()
    ],
    "Accuracy SD": [
        recording_cv_results[
            "test_accuracy"
        ].std(),
        subject_cv_results[
            "test_accuracy"
        ].std()
    ],
    "F1 mean": [
        recording_cv_results[
            "test_f1"
        ].mean(),
        subject_cv_results[
            "test_f1"
        ].mean()
    ],
    "F1 SD": [
        recording_cv_results[
            "test_f1"
        ].std(),
        subject_cv_results[
            "test_f1"
        ].std()
    ],
    "ROC-AUC mean": [
        recording_cv_results[
            "test_roc_auc"
        ].mean(),
        subject_cv_results[
            "test_roc_auc"
        ].mean()
    ],
    "ROC-AUC SD": [
        recording_cv_results[
            "test_roc_auc"
        ].std(),
        subject_cv_results[
            "test_roc_auc"
        ].std()
    ]
})

display(
    cv_summary.round(3)
)


In [ ]:
fold_results = pd.DataFrame({
    "Recording-level": recording_cv_results[
        "test_accuracy"
    ],
    "Subject-level": subject_cv_results[
        "test_accuracy"
    ]
})

fold_results_long = fold_results.melt(
    var_name="Split method",
    value_name="Accuracy"
)

plt.figure(figsize=(7, 5))

sns.boxplot(
    data=fold_results_long,
    x="Split method",
    y="Accuracy"
)

sns.stripplot(
    data=fold_results_long,
    x="Split method",
    y="Accuracy",
    size=8
)

plt.ylim(0, 1)
plt.title("5-fold CV Accuracy")
plt.show()


### ミニ演習5：Cross-validationを考える

1. Mean Accuracyはどちらが高いですか？
2. ROC-AUCはどうですか？
3. Fold間のばらつきはありますか？
4. subject数が少ないことは評価の安定性に影響しそうですか？

答え：

1.
2.
3.
4.


---
# 18. 原著論文と比較する

原著論文では、

- dysphonia measureを比較
- 相関の低い10特徴量を選択
- 特徴量の組合せを探索
- kernel SVMを使用

し、4特徴量の組合せで91.4%のclassification performanceを報告しています。

今回の解析は、

- Logistic Regression
- 全特徴量
- recording-level / subject-level evaluationの比較

を行っています。

したがって、

> **論文と同じAccuracyを出すこと**

が目的ではありません。

今日の目的は、

> **AI性能は、modelだけでなくevaluation designにも影響される**

ことを理解することです。


---
# 19. 追加チャレンジ：kernel SVMを試す

時間に余裕がある人は、原著論文と同じ種類のkernel modelである
RBF kernel SVMを試します。

ただし、

- feature selection方法
- hyperparameter tuning
- validation method

は原著論文と同一ではありません。

したがって、これは完全再現ではなく、
**追加の比較実験**です。


In [ ]:
svm_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            kernel="rbf",
            probability=True,
            C=1.0,
            gamma="scale"
        )
    )
])

svm_recording_cv = cross_validate(
    svm_model,
    X,
    y,
    cv=recording_cv,
    scoring=scoring
)

svm_subject_cv = cross_validate(
    svm_model,
    X,
    y,
    groups=groups,
    cv=subject_cv,
    scoring=scoring
)

svm_comparison = pd.DataFrame({
    "Evaluation": [
        "Recording-level CV",
        "Subject-level CV"
    ],
    "Accuracy": [
        svm_recording_cv[
            "test_accuracy"
        ].mean(),
        svm_subject_cv[
            "test_accuracy"
        ].mean()
    ],
    "F1": [
        svm_recording_cv[
            "test_f1"
        ].mean(),
        svm_subject_cv[
            "test_f1"
        ].mean()
    ],
    "ROC-AUC": [
        svm_recording_cv[
            "test_roc_auc"
        ].mean(),
        svm_subject_cv[
            "test_roc_auc"
        ].mean()
    ]
})

display(
    svm_comparison.round(3)
)


### 追加考察

1. Logistic RegressionとSVMで結果は変わりましたか？
2. SVMに変えればevaluation designの問題は解決しますか？
3. model selectionとevaluation designは別の問題だと言えますか？

答え：

1.
2.
3.


---
# 20. ミニ研究レポート

最後に、今日の解析を短くまとめてください。

## 研究目的

声の特徴から：

## Recording-level splitの結果

- Accuracy：
- F1：
- ROC-AUC：

## subject overlap

TrainとTestに共通したsubject数：

## Subject-level splitの結果

- Accuracy：
- F1：
- ROC-AUC：

## Cross-validationからわかったこと

## どちらの評価方法が未知の人への性能評価として適切だと思うか

答え：

理由：

## この研究の限界

## 次に行いたい解析

例：

- より多くのsubjectを集める
- 外部データセットで評価する
- feature selectionを行う
- SVMのhyperparameterを調整する
- Leave-One-Subject-Out CVを試す

あなたの案：


---
# 21. まとめ

今日の午後は、公開されたParkinson病音声データを使って、

1. 研究背景を理解する
2. データ構造を確認する
3. 同じsubjectから複数録音があることを確認する
4. 音声特徴量を可視化する
5. Logistic Regressionで分類する
6. recording-level random splitで性能評価する
7. Train/Testのsubject overlapを確認する
8. subject-level splitで再評価する
9. 5-fold CVで評価を比較する
10. kernel SVMを追加で試す

という流れを体験しました。

---

## 今日一番伝えたいこと

AI研究では、

> **高いAccuracyが出た**

だけでは十分ではありません。

重要なのは、

> **そのTest dataは、本当に未知の対象を表しているか？**

を考えることです。

同じsubjectから複数の測定値がある場合には、

- subject
- patient
- animal
- sample origin
- institution

などの単位を意識してdata splitを設計する必要があります。

---

## 2日間の午後を振り返る

### 1日目午後

**公開RNA-seqデータから候補遺伝子を見つける**

データ解析から生物学的仮説へ。

### 2日目午後

**高いAI性能を疑ってみる**

モデル開発から信頼できる評価設計へ。

---

## 参考情報

- UCI Machine Learning Repository: Parkinsons Dataset
- Dataset DOI: 10.24432/C59C74
- Little MA et al.
- IEEE Trans Biomed Eng. 2009;56(4):1015–1022
- DOI: 10.1109/TBME.2008.2005954
